In [219]:

from globalVars import TESTFILENAME
TESTFILENAME = 'jacob_1'
print(TESTFILENAME)

jacob_1


In [220]:
# import cv2
# # from google.colab.patches import cv2_imshow


# img = cv2.imread("image.jpg")
# cv2.imshow('image', img)

# Hand Landmarks Detection with MediaPipe Tasks

This notebook shows you how to use MediaPipe Tasks Python API to detect hand landmarks from images.

## Preparation

Let's start with installing MediaPipe.

In [221]:
# !pip install -q mediapipe

Then download an off-the-shelf model bundle. Check out the [MediaPipe documentation](https://developers.google.com/mediapipe/solutions/vision/hand_landmarker#models) for more information about this model bundle.

In [222]:
# !wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

## Visualization utilities

Optionally, you can upload your own image. If you want to do so, uncomment and run the cell below.

## Running inference and visualizing the results

Here are the steps to run hand landmark detection using MediaPipe.

Check out the [MediaPipe documentation](https://developers.google.com/mediapipe/solutions/vision/hand_landmarker/python) to learn more about configuration options that this solution supports.


In [223]:
# STEP 1: Import the necessary modules.
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import json

# STEP 2: Create an HandLandmarker object.
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(base_options=base_options,
                                       num_hands=2)
detector = vision.HandLandmarker.create_from_options(options)

# STEP 3: Load the input image.
image = mp.Image.create_from_file(TESTFILENAME + '.jpg')



In [224]:

def dumper(obj):
    try:
        return obj.toJSON()
    except:
        return obj.__dict__
    


# STEP 4: Detect hand landmarks from the input image.
detection_result = detector.detect(image)

# STEP 5: Process the classification result. In this case, visualize it.
annotated_image = draw_landmarks_on_image(image.numpy_view(), detection_result)

with open(TESTFILENAME + '.json', 'w') as f:
    # print(detection_result.__dict__)
    # print(detection_result.__dict__)
    f.write(json.dumps(detection_result.__dict__, default=dumper, indent=2))
    print(detection_result)
    # f.write(json.dumps(detection_result.__dict__))
# cv2.imshow('window', cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR))
cv2.imwrite(TESTFILENAME + '_labeled.jpg', cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR))
cv2.imwrite(TESTFILENAME + '_out.jpg', cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR))

HandLandmarkerResult(handedness=[[Category(index=0, score=0.8794999122619629, display_name='Right', category_name='Right')]], hand_landmarks=[[NormalizedLandmark(x=0.22876319289207458, y=0.3809930682182312, z=1.162409191124425e-07, visibility=0.0, presence=0.0), NormalizedLandmark(x=0.2441452145576477, y=0.2995920479297638, z=-0.014878506772220135, visibility=0.0, presence=0.0), NormalizedLandmark(x=0.2421587109565735, y=0.21512705087661743, z=-0.01879272423684597, visibility=0.0, presence=0.0), NormalizedLandmark(x=0.2346360683441162, y=0.14256907999515533, z=-0.02233809418976307, visibility=0.0, presence=0.0), NormalizedLandmark(x=0.22676436603069305, y=0.08639070391654968, z=-0.02581936866044998, visibility=0.0, presence=0.0), NormalizedLandmark(x=0.16905812919139862, y=0.1736871600151062, z=0.0018145202193409204, visibility=0.0, presence=0.0), NormalizedLandmark(x=0.13231991231441498, y=0.1131901890039444, z=-0.005099944304674864, visibility=0.0, presence=0.0), NormalizedLandmark(x

True

In [225]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

In [226]:
#@markdown We implemented some functions to visualize the hand landmark detection results. <br/> Run the following cell to activate the functions.

from mediapipe import solutions
from mediapipe.framework.formats import landmark_pb2
import numpy as np

MARGIN = 10  # pixels
FONT_SIZE = 1
FONT_THICKNESS = 1
HANDEDNESS_TEXT_COLOR = (88, 205, 54) # vibrant green

def draw_landmarks_on_image(rgb_image, detection_result):
  hand_landmarks_list = detection_result.hand_landmarks
  handedness_list = detection_result.handedness
  annotated_image = np.copy(rgb_image)

  # Loop through the detected hands to visualize.
  for idx in range(len(hand_landmarks_list)):
    hand_landmarks = hand_landmarks_list[idx]
    handedness = handedness_list[idx]

    # Draw the hand landmarks.
    hand_landmarks_proto = landmark_pb2.NormalizedLandmarkList()
    hand_landmarks_proto.landmark.extend([
      landmark_pb2.NormalizedLandmark(x=landmark.x, y=landmark.y, z=landmark.z) for landmark in hand_landmarks
    ])
    solutions.drawing_utils.draw_landmarks(
      annotated_image,
      hand_landmarks_proto,
      solutions.hands.HAND_CONNECTIONS,
      solutions.drawing_styles.get_default_hand_landmarks_style(),
      solutions.drawing_styles.get_default_hand_connections_style())

    # Get the top left corner of the detected hand's bounding box.
    height, width, _ = annotated_image.shape
    x_coordinates = [landmark.x for landmark in hand_landmarks]
    y_coordinates = [landmark.y for landmark in hand_landmarks]
    text_x = int(min(x_coordinates) * width)
    text_y = int(min(y_coordinates) * height) - MARGIN

    # Draw handedness (left or right hand) on the image.
    cv2.putText(annotated_image, f"{handedness[0].category_name}",
                (text_x, text_y), cv2.FONT_HERSHEY_DUPLEX,
                FONT_SIZE, HANDEDNESS_TEXT_COLOR, FONT_THICKNESS, cv2.LINE_AA)

  return annotated_image

## Download test image

Let's grab a test image that we'll use later. The image is from [Unsplash](https://unsplash.com/photos/mt2fyrdXxzk).

##### Copyright 2023 The MediaPipe Authors. All Rights Reserved.